# Vector DB

In this notebook, we use a containerized version of Chroma DB. To set up, you will need the following:

1. Install [Docker Desktop](https://www.docker.com/products/docker-desktop/) by following the link and Download Docker Desktop for your operating system.
2. In a terminal window, navigate to the folder ./05_src/chromadb/. For example, on Windows, you would use `cd .\05_src\chromadb`.
3. Run the command `docker compose up -d`, which will start the Chroma DB server.

## Downloading Batch Results

In the previous notebook, we had created batch processes. We will start by consulting the status of our batch processes by identifying them throught their descriptions.

In [1]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [2]:
batch_description = 'Pitchfork reviews content embeddings (kashfa_azim) 2025-10-30 15:27:57'

In [3]:
from openai import OpenAI

client = OpenAI()

batch_processes = client.batches.list().to_dict()
batch_info= [
    {'batch_id': batch['id'],
     'description': batch['metadata']['description'],
    'status': batch['status'],
    'request_counts': batch['request_counts'],
    'output_file_id': batch['output_file_id'],
    'input_file_id': batch['input_file_id']}
            for batch in batch_processes['data'] #if batch['metadata']['description'] == batch_description
    ]
batch_info

[{'batch_id': 'batch_6904e7efb1a081909ae0e3dfe1420951',
  'description': 'Pitchfork reviews content embeddings Gibran_Alvarez + 2025-10-31 12:46:18',
  'status': 'completed',
  'request_counts': {'completed': 10000, 'failed': 0, 'total': 10000},
  'output_file_id': 'file-Fa42QupdysXS7bNAVdjByN',
  'input_file_id': 'file-AxWGX2hLamC7YQwRkkNpTh'},
 {'batch_id': 'batch_6904e7ef769c8190b0e702d324aa42bf',
  'description': 'Pitchfork reviews content embeddings Gibran_Alvarez + 2025-10-31 12:46:18',
  'status': 'completed',
  'request_counts': {'completed': 10000, 'failed': 0, 'total': 10000},
  'output_file_id': 'file-EihZFAZuZhcMtACizajS9F',
  'input_file_id': 'file-My4B1pwancsAQM8EoBpqtH'},
 {'batch_id': 'batch_6904e7eebc1c819094d4a54ab791d144',
  'description': 'Pitchfork reviews content embeddings Gibran_Alvarez + 2025-10-31 12:46:18',
  'status': 'completed',
  'request_counts': {'completed': 10000, 'failed': 0, 'total': 10000},
  'output_file_id': 'file-KJMtZHS319RqF4gYciTWfH',
  'inpu

When the status of the batches is complete, we can query the `output_file_id` where their results will be stored.

More generally, we will require the original text and the embeddings of that original text mapped through the `custom_id`.

In [4]:
batch_complete = [
    batch  for batch in batch_info if batch['status'] == 'completed'
]
batch_complete

[{'batch_id': 'batch_6904e7efb1a081909ae0e3dfe1420951',
  'description': 'Pitchfork reviews content embeddings Gibran_Alvarez + 2025-10-31 12:46:18',
  'status': 'completed',
  'request_counts': {'completed': 10000, 'failed': 0, 'total': 10000},
  'output_file_id': 'file-Fa42QupdysXS7bNAVdjByN',
  'input_file_id': 'file-AxWGX2hLamC7YQwRkkNpTh'},
 {'batch_id': 'batch_6904e7ef769c8190b0e702d324aa42bf',
  'description': 'Pitchfork reviews content embeddings Gibran_Alvarez + 2025-10-31 12:46:18',
  'status': 'completed',
  'request_counts': {'completed': 10000, 'failed': 0, 'total': 10000},
  'output_file_id': 'file-EihZFAZuZhcMtACizajS9F',
  'input_file_id': 'file-My4B1pwancsAQM8EoBpqtH'},
 {'batch_id': 'batch_6904e7eebc1c819094d4a54ab791d144',
  'description': 'Pitchfork reviews content embeddings Gibran_Alvarez + 2025-10-31 12:46:18',
  'status': 'completed',
  'request_counts': {'completed': 10000, 'failed': 0, 'total': 10000},
  'output_file_id': 'file-KJMtZHS319RqF4gYciTWfH',
  'inpu

Before we download all results, examine the response of the file API:

In [5]:
response = client.files.content(batch_complete[0]['output_file_id'])
text_response = response.text
lines = text_response.split('\n')
print(lines[0])


{"id": "batch_req_6904f39910a08190ae0ff81dc260a183", "custom_id": "22703_1_0", "response": {"status_code": 200, "request_id": "b3d6da58121ca2cfd902f447c287a966", "body": {"object": "list", "data": [{"object": "embedding", "index": 0, "embedding": [-0.029599756, 0.013184268, 0.0018101503, -0.035573065, 0.011109172, -0.0063290414, 0.02134384, 0.015592861, 0.013532587, 0.03047426, 0.026205491, 0.015355707, -0.023003915, 0.0069219256, -0.047519688, 0.006825582, -0.023967354, -0.027672881, -0.034654096, 0.004872769, -0.017000962, -0.0015035179, -0.036403105, 0.014614602, -0.021877436, 0.010760852, -0.028428808, 0.03299402, 0.002769882, 0.008226272, -0.02470846, -0.029155092, 0.06343863, 0.022722296, 0.010308778, -0.023982175, 0.0062956917, -0.010405122, 0.014273693, 0.009878937, 0.025716363, -0.012435751, 0.026042448, -0.0008536609, 0.009878937, 0.00533596, -0.0030700297, 0.013406599, 0.03124501, 0.06296433, -0.0028625203, 0.0086116465, -0.044140246, 0.013458476, -0.021299373, -0.037114564,

For our results database, we will need to map the original text to their embeddings. 

In [6]:
import json 

def get_text_and_embeddings(batch):
    embedding_lines =  get_content_from_file(batch, 'output_file_id')
    text_lines = get_content_from_file(batch, 'input_file_id')
    return embedding_lines, text_lines

def get_content_from_file(batch, key):
    file = client.files.content(batch[key])
    text = file.text
    lines = text.split('\n')
    content_lines = [json.loads(line) for line in lines if line.strip()]
    return content_lines


Notice that the response is also a .jsonl file. Therefore, we can process it line-by-line and use the `custom_id` to map to the original document chunk.

The function below:

- Creates a dictionary, `text_dict`, with keys given by each `custom_id` and value equal to the text.
- Iterate over all embedding items and use the dictionary defined above to map the embeddings to their input text.

In [7]:
def create_chroma_inputs(embedding_lines, text_lines):
    chroma_inputs = []
    text_dict = {item['custom_id']: item['body']['input'] for item in text_lines}
    for embed_item in embedding_lines:
        custom_id = embed_item['custom_id']
        text = text_dict.get(custom_id, "")
        chroma_input = {
            'id': embed_item['custom_id'],
            'embedding': embed_item['response']['body']['data'][0]['embedding'],
            'text': text
        }
        chroma_inputs.append(chroma_input)
    return chroma_inputs

A couple of functions to control the logic flow:

In [8]:
from tqdm import tqdm

def process_batch_for_chromadb(batch):
    embedding_lines, text_lines = get_text_and_embeddings(batch)
    chroma_inputs = create_chroma_inputs(embedding_lines, text_lines)
    return chroma_inputs

def process_batches_for_chromadb(batches):
    all_chroma_inputs = []
    for batch in tqdm(batches, desc="Processing batches"):
        chroma_inputs = process_batch_for_chromadb(batch)
        all_chroma_inputs.extend(chroma_inputs)
    return all_chroma_inputs

Now, we can create our input dictionaries.

In [9]:
chroma_inputs = process_batches_for_chromadb(batch_complete)

Processing batches: 100%|██████████| 10/10 [03:56<00:00, 23.68s/it]


In [10]:
import os
with open('../../05_src/documents/chroma_inputs.jsonl', 'r') as f:
    lines = f.readlines()
    chroma_inputs = [json.loads(line) for line in lines if line.strip()]

In [11]:
chroma_inputs[1]

{'id': '22703_1_1800',
 'embedding': [0.012155707,
  0.013781581,
  0.031750552,
  -0.018452132,
  0.009716896,
  0.02053816,
  0.03273221,
  0.008091022,
  0.0012913042,
  0.013796919,
  -0.015116024,
  0.0030197536,
  -0.01212503,
  0.032885596,
  -0.037640512,
  0.030400772,
  -0.007216732,
  -0.0026938121,
  -0.02337577,
  -0.011657208,
  0.0042065647,
  0.03451147,
  -0.007009663,
  0.0150776785,
  -0.013758573,
  0.029419111,
  0.021351097,
  0.02639744,
  0.035155684,
  0.0010305508,
  -0.0022585841,
  -0.03647479,
  0.030278064,
  0.0019882442,
  -0.025507811,
  -0.026811577,
  0.009325766,
  -0.046138003,
  -0.052058637,
  -0.021427788,
  0.0596665,
  0.023022985,
  0.012937967,
  0.03267086,
  0.005564016,
  0.010184718,
  0.0057672504,
  -0.009210728,
  0.04475754,
  0.06003462,
  -0.03527839,
  0.010729233,
  -0.010867279,
  0.042088658,
  0.012608191,
  -0.058654163,
  -0.042794224,
  -0.01563753,
  -0.0032632514,
  0.0023045994,
  0.016887613,
  0.016090015,
  0.003338026

# Load Embeddings to Chroma

In [12]:

import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
import os


def setup_collection(chroma_url:str="http://localhost:8000",
                     collection_name: str = "pitchfork_reviews"):
    chroma_client = chromadb.HttpClient(host=chroma_url)
    collections = chroma_client.list_collections()
    if collection_name in [col.name for col in collections]:
        chroma_client.delete_collection(name=collection_name)

    collection = chroma_client.create_collection(
        name=collection_name,
        embedding_function=OpenAIEmbeddingFunction(
            api_key = os.getenv("OPENAI_API_KEY"),
            model_name="text-embedding-3-small")
        )
    return collection

def load_embeddings_to_db(chroma_inputs:list[dict], 
                          collection_name:str,
                          chroma_url:str="http://localhost:8000",
                          batch_size:int= 1000
                          ):

    
    collection = setup_collection(chroma_url=chroma_url, collection_name=collection_name)

    for i in tqdm(range(0, len(chroma_inputs), batch_size)):
        batch = chroma_inputs[i:i + batch_size]
        collection.add(
            documents=[item['text'] for item in batch],
            embeddings=[item['embedding'] for item in batch],
            ids=[item['id'] for item in batch]
        )


In [13]:
vector_db_client_url:str="http://localhost:8000"
load_embeddings_to_db(chroma_inputs=chroma_inputs,
                      collection_name="pitchfork_reviews",
                      chroma_url=vector_db_client_url, 
                      batch_size=1000)

100%|██████████| 49/49 [00:53<00:00,  1.08s/it]


# Additional Details

We will use a simple database to store additional details about the reviews. In this case, we load the jsonl files, and use pandas to create a few tables in a sql database. The connection string to the database is included in the .secrets file.

In [14]:
import json

def load_jsonl(file:str):
    data = []
    with open(file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

In [19]:
import pandas as pd
import sqlalchemy as sa
import os

doc_folder = "../../05_src/documents/"
tables = ["artists", "reviews", "labels", "genres"]

def upload_tables_to_sql(tables:list[str], doc_folder:str):
    engine = sa.create_engine(os.getenv("SQL_URL"))
    for table_name in tables:
        file_path = os.path.join(doc_folder, f"pitchfork_{table_name}.jsonl")
        data = load_jsonl(file_path)
        df = pd.DataFrame(data)
        with engine.connect() as conn:
            df.to_sql(table_name, conn, if_exists='replace', index=False)
        print(f"Loaded {df.shape} records from {file_path}")

upload_tables_to_sql(tables=tables, doc_folder=doc_folder)

Loaded (18831, 2) records from ../../05_src/documents/pitchfork_artists.jsonl
Loaded (18393, 13) records from ../../05_src/documents/pitchfork_reviews.jsonl
Loaded (20190, 2) records from ../../05_src/documents/pitchfork_labels.jsonl
Loaded (22680, 2) records from ../../05_src/documents/pitchfork_genres.jsonl


In [20]:
def additional_details(review_id:str):
    import sqlalchemy as sa
    import pandas as pd
    import os

    engine = sa.create_engine(os.getenv("SQL_URL"))
    query = f"""
    SELECT r.reviewid,
		r.title,
		r.artist,
		r.score,
		g.genre
    FROM reviews AS r
    LEFT JOIN genres as g
	    ON r.reviewid = g.reviewid
    WHERE r.reviewid = '{review_id}'
    """
    with engine.connect() as conn:
        result = pd.read_sql(query, conn)
    if not result.empty:
        row = result.iloc[0]
        details = {
            "reviewid": row['reviewid'],
            "album": row['title'],
            "score": row['score'],
            "artist": row['artist']
        }
        return details
    else:
        return {}
    
def get_reviewid_from_custom_id(custom_id:str):
    return custom_id.split('_')[0]

# Prompt Generator

Here we create a prompt with the context gathered through the different data operations.

In [21]:
chroma = chromadb.HttpClient(host=vector_db_client_url)
collection = chroma.get_collection(name="pitchfork_reviews", 
                                   embedding_function=OpenAIEmbeddingFunction(
                                       api_key = os.getenv("OPENAI_API_KEY"),
                                       model_name="text-embedding-3-small")
                                   )


In [22]:
collection.query(
    query_texts=["A great album with stunning vocals and production."],
    n_results=3
)

{'ids': [['398_15632_1801', '4428_16669_1802', '6881_16771_3609']],
 'distances': [[1.0193821, 1.0360923, 1.0362467]],
 'embeddings': None,
 'metadatas': [[None, None, None]],
 'documents': [['vocals take on an even more tranquil quality than previous offerings, just begging to grace     a pair of headphones.  In fact, the subtle variations in vocal renderings between the three songs are the     only true variants within the album, which follows logically given no-one has any misconceptions about     where Azure Ray\'s bread is buttered.          "The Love of Two", the only non-album track, is the most atmospheric, as well as most memorable.  An     incessant vinyl scratch murmurs beneath a subtle vocal tonality reminiscent of Mazzy Star, which, distant     and haunted, provides an excellent equipoise to the visceral production.  A "Bleed Version" of the     album-inclusive "We Are Mice" rounds out the disc with sparse piano backing and choral vocals that I     swear could only be desc

In [23]:
def get_context_data(query:str, collection:chromadb.api.models.Collection, top_n:int):
    results = collection.query(
        query_texts=[query],
        n_results=top_n
    )
    context_data = []
    for idx, custom_id in enumerate(results['ids'][0]):
        review_id = get_reviewid_from_custom_id(custom_id)
        details = additional_details(review_id)
        details['text'] = results['documents'][0][idx]
        context_data.append(details)
    return context_data

def generate_prompt(query:str, collection:chromadb.api.models.Collection, top_n:int):
    context_data = get_context_data(query, collection, top_n)
    prompt = f"Given a query, provide a detailed response using the context from relevant Pitchfork reviews. The context will contain references to {top_n} album reviews.\n\n"
    prompt += f"The score is numeric and its scale is from 0 to 10, with 10 being the highest rating. Any album with a score greater than 8.0 is considered a must-listen; album with a score greater than 6.5 is good.\n\n"
    prompt += f"<query>{query}</query>\n\n"
    prompt += "<context>\n"
    for k, context in enumerate(context_data):
        prompt += f"<album {k}>\n"
        prompt += f"- Album Title: {context.get('album', 'N/A')}\n" 
        prompt += f"- Album Artist: {context.get('artist', 'N/A')}\n"
        prompt += f"- Album Score: {context.get('score', 'N/A')}\n"
        prompt += f"- Review Quote: {context.get('text', 'N/A')}\n"
        prompt += f"</album {k}>\n\n"
    prompt += "</context>\n\n"
    prompt += "\nBased on the context and nothing else, provide a detailed response to the query."
    return prompt

def generate_response(query:str, collection:chromadb.api.models.Collection, top_n:int=1):
    prompt = generate_prompt(query, collection, top_n)
    print("Generated Prompt:\n", prompt)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that provides information based on Pitchfork reviews."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=500,
        temperature=0.7
    )
    return response.choices[0].message.content

# Query

We can now use chroma's similarity function to query the database. Notice that the query itself needs to be converted to embeddings, so we must provide an `embedding_function`. In this case, we use `OpenAIEmbeddingFunction()` to get compatible embeddings using model `text-embedding-3-small`.

In [26]:
response = generate_response("What are some highly rated albums by emerging indie artists?", collection, 3)

Generated Prompt:
 Given a query, provide a detailed response using the context from relevant Pitchfork reviews. The context will contain references to 3 album reviews.

The score is numeric and its scale is from 0 to 10, with 10 being the highest rating. Any album with a score greater than 8.0 is considered a must-listen; album with a score greater than 6.5 is good.

<query>What are some highly rated albums by emerging indie artists?</query>

<context>
<album 0>
- Album Title: the twilight saga: new moon ost
- Album Artist: various artists
- Album Score: 5.4
- Review Quote: Sensitive young men who avoid sunlight and the gloomy misfit young women who adore them: For all the gasps that greeted Twilight author Stephanie Meyer's recent embrace of indie rock, the parallels between the two are obvious enough. On different scales, each has seen its financial fortunes rise the past few years as well. The music industry's troubles are widely known, but indie's stock continues to climb. Phoenix

In [27]:
print(response)

If you're looking for highly rated albums by emerging indie artists, one standout is **"Conductor" by The Comas**, which received a score of **8.0** from Pitchfork. This album embodies the indie mindset, exploring the tension between artistic integrity and commercial viability. The Comas navigate the space between indie and mainstream, creating music that resonates on both personal and broader levels. Their sound reflects a blend of sincerity and accessibility, making "Conductor" a compelling listen for fans of the genre.

Another notable mention is the **"Live at KEXP Volume 2"** compilation, which scored **6.9**. This album features a variety of emerging indie acts drawn from over 370 in-studio performances at KEXP. It showcases a diverse range of styles, including indie rock, pop, rap, and folk, making it an excellent representation of the current indie music landscape. While it may not be as highly rated as "Conductor," it serves as a great introduction to several emerging artists 

**Note**: Try changing the top_n parameter to 1 and re-run the query. 